# PG-Stack · Step 5 — Label efficiency & cross-hospital generalisation of graph ugly-duckling priors

**Hypothesis.** Label-free graph-outlier scores (LOF / Mahalanobis / Isolation Forest per patient; AUROC ≈ 0.87 with *zero* labels) give little extra when 393 labelled cancers are available (Step 4c), **but help a lot when labels are scarce or the hospital is new.**

**Models** (LightGBM, same recipe everywhere)
| Name | Features |
|---|---|
| `M0_lesion` | lesion features only |
| `M1_global` | + global patient normalisation (≈ tab_pnorm) |
| `M2_prior` | + **graph prior** (label-free outlier scores) |
| `M3_global_prior` | + both |
| `U_zero_label` | no training at all: average rank of 3 label-free outlier scores |

**Experiment A — learning curve.** Train on 5 / 10 / 25 / 50 / 100 % of training patients (stratified, 3 repeats below 100 %), same 5 patient folds; test always on the full held-out fold.
**Experiment B — leave-one-hospital-out (LOHO).** Train on 6 hospitals, test on the 7th (`attribution` column).

Images are not used here (the CNN/DINOv2 heads are tied to the original folds) — noted as a limitation.
CPU is enough. Runtime ≈ 70–90 min; results are saved as they finish (resumable).

In [ ]:
import os, sys, glob, time, gc, warnings
import numpy as np, pandas as pd, lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
COMP_DIR = "/kaggle/input/competitions/isic-2024-challenge"
SEED = 42
FRACTIONS = [0.05, 0.10, 0.25, 0.50, 1.00]
REPEATS = 3                 # repeats for fractions < 1
N_BAGS = 5
NEG_PER_POS = 20            # negatives sampled per positive (same rule at every data size)
N_BOOT = 200
MIN_POS_HOSPITAL = 10       # hospitals with fewer cancers are reported but not tested

def locate(name):
    if os.path.exists(f"{WORK}/{name}"): return f"{WORK}/{name}"
    for d in range(1, 5):
        hits = glob.glob("/kaggle/input/" + "*/" * d + name)
        if hits: return hits[0]
    return None
paths = {n: locate(n) for n in ["oof_step3.csv", "emb_pca32.npy", "pgstack_eval.py"]}
assert all(paths.values()), f"missing: {[k for k, v in paths.items() if v is None]} (upload your backups or re-run Steps 1-3)"
sys.path.insert(0, os.path.dirname(paths["pgstack_eval.py"]))
from pgstack_eval import *
print(paths)

## 1. Lesion features (identical to Steps 1 / 4b / 4c)

In [ ]:
oof = pd.read_csv(paths["oof_step3.csv"])
df = pd.read_csv(f"{COMP_DIR}/train-metadata.csv", low_memory=False)
df = oof[["isic_id", "fold"]].merge(df, on="isic_id", how="left")
assert (df.isic_id.values == oof.isic_id.values).all()
emb = np.load(paths["emb_pca32.npy"]).astype(np.float32)
assert len(emb) == len(oof)

LEAK = ["lesion_id", "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
        "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]
NUM_RAW = [c for c in df.columns if (c.startswith("tbp_lv_") or c in ["age_approx", "clin_size_long_diam_mm"])
           and c not in LEAK and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple"]

def add_lesion_features(d):
    d = d.copy(); eps = 1e-6
    d["age_approx"] = pd.to_numeric(d["age_approx"], errors="coerce")
    d["lesion_size_ratio"] = d.tbp_lv_minorAxisMM / (d.clin_size_long_diam_mm + eps)
    d["lesion_shape_index"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM ** 2 + eps)
    d["hue_contrast"] = (d.tbp_lv_H - d.tbp_lv_Hext).abs()
    d["luminance_contrast"] = (d.tbp_lv_L - d.tbp_lv_Lext).abs()
    d["lesion_color_difference"] = np.sqrt(d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2)
    d["border_complexity"] = d.tbp_lv_norm_border + d.tbp_lv_symm_2axis
    d["color_uniformity"] = d.tbp_lv_color_std_mean / (d.tbp_lv_radial_color_std_max + eps)
    d["position_distance_3d"] = np.sqrt(d.tbp_lv_x**2 + d.tbp_lv_y**2 + d.tbp_lv_z**2)
    d["perimeter_to_area_ratio"] = d.tbp_lv_perimeterMM / (d.tbp_lv_areaMM2 + eps)
    d["area_to_perimeter_ratio"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM + eps)
    d["lesion_visibility_score"] = d.tbp_lv_deltaLBnorm + d.tbp_lv_norm_color
    d["symmetry_border_consistency"] = d.tbp_lv_symm_2axis * d.tbp_lv_norm_border
    d["color_shape_composite"] = (d.tbp_lv_color_std_mean + d.tbp_lv_area_perim_ratio + d.tbp_lv_symm_2axis) / 3
    d["log_area"] = np.log1p(d.tbp_lv_areaMM2)
    d["normalized_lesion_size"] = d.clin_size_long_diam_mm / (d.age_approx + eps)
    d["mean_hue_difference"] = (d.tbp_lv_H + d.tbp_lv_Hext) / 2
    d["std_dev_contrast"] = np.sqrt((d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2) / 3)
    return d

ENG = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast", "lesion_color_difference",
       "border_complexity", "color_uniformity", "position_distance_3d", "perimeter_to_area_ratio",
       "area_to_perimeter_ratio", "lesion_visibility_score", "symmetry_border_consistency",
       "color_shape_composite", "log_area", "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast"]
df = add_lesion_features(df)
LESION_NUM = sorted(set(NUM_RAW + ENG))
for c in CAT_COLS: df[c] = df[c].fillna("NA").astype("category")
y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
N_FOLDS = int(folds.max()) + 1
POS_COLS = ["tbp_lv_x", "tbp_lv_y", "tbp_lv_z", "position_distance_3d"]
TAB_COLS = [c for c in LESION_NUM if c not in POS_COLS]        # appearance/shape only, for the tabular graph
print(len(df), "lesions |", len(LESION_NUM), "lesion features")

In [ ]:
df["hospital"] = df["attribution"].fillna("unknown").astype(str)
print(df.groupby("hospital").agg(lesions=("isic_id", "size"), cancers=("target", "sum"),
                                 patients=("patient_id", "nunique")).sort_values("cancers", ascending=False))

## 2. Global patient normalisation (label-free)

In [ ]:
g = df.groupby("patient_id"); GLOBAL = []
for c in LESION_NUM:
    df[f"{c}_pnorm"] = (df[c] - g[c].transform("mean")) / (g[c].transform("std") + 1e-6); GLOBAL.append(f"{c}_pnorm")
df["patient_n_lesions"] = g["isic_id"].transform("size"); GLOBAL.append("patient_n_lesions")

## 3. Graph prior: label-free outlier scores per patient
Tabular graph (shape/colour) and appearance graph (DINOv2): kNN distance and LOF at k = 5/10/20; per-patient Mahalanobis and Isolation Forest; site-adjusted body density. **No labels are used**, so these can be computed for any new patient or hospital.

In [ ]:
X = df[LESION_NUM].astype(np.float32); X = X.fillna(X.median()).values
tab_idx = [LESION_NUM.index(c) for c in TAB_COLS]
TAB = StandardScaler().fit_transform(X[:, tab_idx]).clip(-6, 6).astype(np.float32)
TAB10 = PCA(10, random_state=SEED).fit_transform(TAB).astype(np.float32)
XYZ = df[["tbp_lv_x", "tbp_lv_y", "tbp_lv_z"]].fillna(0).values.astype(np.float32)
SP = {"tab": TAB, "app": emb, "pos": XYZ}
n = len(df)
GP = {f"gp_{s}_k{k}_{t}": np.full(n, np.nan, np.float32) for s in SP for k in [5, 10, 20] for t in ["dist", "lof"]}
GP["gp_mahal"] = np.full(n, np.nan, np.float32); GP["gp_iforest"] = np.full(n, np.nan, np.float32)

def lof(dist, idx, k):
    d, i = dist[:, :k], idx[:, :k]; reach = np.maximum(d, d[:, -1][i])
    lrd = 1.0 / (reach.mean(1) + 1e-6); return lrd[i].mean(1) / lrd

t0 = time.time()
for j, (p, rows) in enumerate(pd.Series(np.arange(n)).groupby(pid).indices.items()):
    r = np.asarray(rows); m = len(r)
    if m < 6: continue
    km = min(20, m - 1)
    for s, Z in SP.items():
        dist, idx = NearestNeighbors(n_neighbors=km + 1).fit(Z[r]).kneighbors(Z[r]); dist, idx = dist[:, 1:], idx[:, 1:]
        for k in [5, 10, 20]:
            kk = min(k, km); GP[f"gp_{s}_k{k}_dist"][r] = dist[:, :kk].mean(1); GP[f"gp_{s}_k{k}_lof"][r] = lof(dist, idx, kk)
    if m >= 15:
        Zp = TAB10[r]; dz = Zp - Zp.mean(0); C = np.cov(Zp, rowvar=False) + 1e-3 * np.eye(10)
        GP["gp_mahal"][r] = np.sqrt(np.einsum("ij,jk,ik->i", dz, np.linalg.inv(C), dz))
    if m >= 10:
        Zi = np.c_[TAB10[r], emb[r, :10]]
        GP["gp_iforest"][r] = -IsolationForest(n_estimators=100, random_state=SEED).fit(Zi).score_samples(Zi)
    if j % 200 == 0: print(f"  patient {j} | {time.time()-t0:.0f}s")
for k, v in GP.items(): df[k] = v
key = df.patient_id.astype(str) + "|" + df.anatom_site_general.astype(str)
df["gp_pos_k10_dist_siteadj"] = df["gp_pos_k10_dist"] / (df.groupby(key)["gp_pos_k10_dist"].transform("median") + 1e-6)
PRIOR = list(GP.keys()) + ["gp_pos_k10_dist_siteadj"]
print(f"graph prior: {len(PRIOR)} features in {time.time()-t0:.0f}s")

# zero-label score: average percentile rank of the 3 strongest label-free scores
U = np.nanmean(np.stack([pd.Series(df[c]).rank(pct=True).values for c in ["gp_tab_k10_lof", "gp_mahal", "gp_iforest"]]), 0)
U = np.nan_to_num(U, nan=0.5)
print("zero-label U score: AUROC", round(roc_auc_score(y, U), 4), "| pAUC80", round(pauc_80(y, U), 4))

## 4. Shared LightGBM trainer

In [ ]:
BASE_F = LESION_NUM + CAT_COLS
FSETS = {"M0_lesion": BASE_F, "M1_global": BASE_F + GLOBAL,
         "M2_prior": BASE_F + PRIOR, "M3_global_prior": BASE_F + GLOBAL + PRIOR}
LGB_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7, min_child_samples=40,
                  feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0,
                  n_estimators=300, verbosity=-1)
XALL = {m: df[f] for m, f in FSETS.items()}

def fit_predict(model, tr_rows, te_rows, seed):
    pos, neg = tr_rows[y[tr_rows] == 1], tr_rows[y[tr_rows] == 0]
    Xte = XALL[model].iloc[te_rows]; pred = np.zeros(len(te_rows))
    for b in range(N_BAGS):
        rng = np.random.default_rng(seed + b)
        sub = np.r_[pos, rng.choice(neg, min(len(neg), NEG_PER_POS * len(pos)), replace=False)]
        m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed + b).fit(XALL[model].iloc[sub], y[sub])
        pred += m.predict_proba(Xte)[:, 1] / N_BAGS
    return pred

pat = df.groupby("patient_id").agg(fold=("fold", "first"), has_pos=("target", "max"))
rows_of = pd.Series(np.arange(n)).groupby(pid).indices

## 5. Experiment A — learning curve

In [ ]:
LC_PATH = f"{WORK}/step5_learning_curve.npz"
LC = dict(np.load(LC_PATH)) if os.path.exists(LC_PATH) else {}
t0 = time.time()
for q in FRACTIONS:
    for rep in range(1 if q == 1.0 else REPEATS):
        if all(f"{m}|{q}|{rep}" in LC for m in FSETS): continue
        preds = {m: np.zeros(n, np.float32) for m in FSETS}
        for f in range(N_FOLDS):
            rng = np.random.default_rng(SEED + 7919 * rep + 101 * f + int(q * 1000))
            cand = pat[pat.fold != f]; chosen = []
            for hp in [0, 1]:                                            # stratified: keep the cancer-patient share
                ids = cand.index[cand.has_pos == hp].values
                chosen += list(rng.choice(ids, max(1, int(round(q * len(ids)))), replace=False))
            tr = np.concatenate([rows_of[p] for p in chosen]); te = np.where(folds == f)[0]
            for m in FSETS: preds[m][te] = fit_predict(m, tr, te, SEED + 13 * f + 1000 * rep)
        for m in FSETS: LC[f"{m}|{q}|{rep}"] = preds[m]
        np.savez_compressed(LC_PATH, **LC)
        print(f"q={q:.2f} rep={rep}: " + "  ".join(f"{m}={pauc_80(y, preds[m]):.4f}" for m in FSETS) +
              f"   ({(time.time()-t0)/60:.1f} min)")

In [ ]:
rows = []
for q in FRACTIONS:
    reps = range(1 if q == 1.0 else REPEATS)
    for m in FSETS:
        v = [pauc_80(y, LC[f"{m}|{q}|{r}"]) for r in reps]
        rows.append(dict(fraction=q, model=m, pauc80=np.mean(v), sd=np.std(v)))
lc = pd.DataFrame(rows)
lc_tab = lc.pivot(index="fraction", columns="model", values="pauc80").round(4)
lc_tab["U_zero_label"] = round(pauc_80(y, U), 4)
lc_tab["gain_prior_over_global (M3-M1)"] = (lc_tab["M3_global_prior"] - lc_tab["M1_global"]).round(4)
lc_tab["gain_prior_over_lesion (M2-M0)"] = (lc_tab["M2_prior"] - lc_tab["M0_lesion"]).round(4)
print(lc_tab.to_string())

In [ ]:
# paired patient bootstrap, averaged over repeats; Holm across the 5 fractions for the primary test (M3 vs M1)
boot = PatientBootstrap(pid, n_boot=N_BOOT, seed=SEED)
picks = [idx for idx in boot.samples if y[idx].sum() > 0]
def paired_curve(a, b, q):
    reps = range(1 if q == 1.0 else REPEATS)
    d = [np.mean([pauc_80(y[i], LC[f"{a}|{q}|{r}"][i]) - pauc_80(y[i], LC[f"{b}|{q}|{r}"][i]) for r in reps]) for i in picks]
    d = np.array(d); return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), (d <= 0).mean()

def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.zeros(len(p)); run = 0
    for rank, i in enumerate(o): run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
    return adj

t0 = time.time(); tests = []
for a, b, fam in [("M3_global_prior", "M1_global", "PRIMARY"), ("M2_prior", "M0_lesion", "secondary")]:
    block = []
    for q in FRACTIONS:
        d, lo, hi, p = paired_curve(a, b, q)
        block.append(dict(family=fam, comparison=f"{a} vs {b}", fraction=q, delta=round(d, 4), ci95=f"[{lo:+.4f}, {hi:+.4f}]", p=round(p, 4)))
    adj = holm([r["p"] for r in block])
    for r, pa in zip(block, adj): r["p_holm"] = round(pa, 4); r["sig_0.05"] = pa < 0.05
    tests += block
lc_tests = pd.DataFrame(tests); print(lc_tests.to_string(index=False)); print(f"({time.time()-t0:.0f}s)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
style = {"M0_lesion": ("lesion only", "#8c8c8c", "o"), "M1_global": ("+ global context", "#1f77b4", "s"),
         "M2_prior": ("+ graph prior", "#2ca02c", "^"), "M3_global_prior": ("+ global + graph prior", "#d62728", "D")}
for m, (lab, colr, mk) in style.items():
    d = lc[lc.model == m]
    ax.errorbar(d.fraction * 100, d.pauc80, yerr=d.sd, label=lab, color=colr, marker=mk, capsize=3)
ax.axhline(pauc_80(y, U), color="#9467bd", ls="--", label="zero-label graph score")
ax.set_xscale("log"); ax.set_xticks([5, 10, 25, 50, 100]); ax.set_xticklabels(["5%", "10%", "25%", "50%", "100%"])
ax.set_xlabel("share of training patients with labels"); ax.set_ylabel("pAUC above 80% TPR (max 0.2)")
ax.set_title("Label efficiency (5-fold patient CV, test on full held-out fold)"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{WORK}/fig_learning_curve.png", dpi=200); plt.show()

## 6. Experiment B — leave-one-hospital-out

In [ ]:
HOSP = df.hospital.values
LOHO_PATH = f"{WORK}/step5_loho.npz"
LH = dict(np.load(LOHO_PATH)) if os.path.exists(LOHO_PATH) else {}
t0 = time.time()
for h in sorted(np.unique(HOSP)):
    if all(f"{m}|{h}" in LH for m in FSETS): continue
    te = np.where(HOSP == h)[0]; tr = np.where(HOSP != h)[0]
    if y[te].sum() == 0: print(f"skip {h}: no cancers"); continue
    for m in FSETS: LH[f"{m}|{h}"] = fit_predict(m, tr, te, SEED + 555).astype(np.float32)
    np.savez_compressed(LOHO_PATH, **LH)
    print(f"{h[:45]:45s} cancers={int(y[te].sum()):3d}  " +
          "  ".join(f"{m}={pauc_80(y[te], LH[f'{m}|{h}']):.4f}" for m in FSETS) + f"  ({(time.time()-t0)/60:.1f} min)")

In [ ]:
rows = []
pooled = {m: np.full(n, np.nan) for m in FSETS}
for h in sorted(np.unique(HOSP)):
    te = np.where(HOSP == h)[0]
    if f"M0_lesion|{h}" not in LH: continue
    r = dict(hospital=h[:40], lesions=len(te), cancers=int(y[te].sum()))
    for m in FSETS:
        p = LH[f"{m}|{h}"]; pooled[m][te] = p
        r[f"{m}_pauc"] = pauc_80(y[te], p); r[f"{m}_auroc"] = roc_auc_score(y[te], p)
    r["U_pauc"] = pauc_80(y[te], U[te]); r["U_auroc"] = roc_auc_score(y[te], U[te])
    rows.append(r)
loho = pd.DataFrame(rows)
show = ["hospital", "lesions", "cancers"] + [f"{m}_pauc" for m in FSETS] + ["U_pauc"]
print(loho[show].round(4).to_string(index=False))
ok = loho.cancers >= MIN_POS_HOSPITAL
print("\nMacro-average over hospitals with >=", MIN_POS_HOSPITAL, "cancers:")
print(loho.loc[ok, [c for c in loho.columns if c.endswith("_pauc") or c.endswith("_auroc")]].mean().round(4).to_string())

In [ ]:
# paired bootstrap on the pooled LOHO predictions (each lesion scored by the model that never saw its hospital)
mask = ~np.isnan(pooled["M0_lesion"])
yy, pp = y[mask], pid[mask]; P = {m: pooled[m][mask] for m in FSETS}
bb = PatientBootstrap(pp, n_boot=N_BOOT * 2, seed=SEED)
SB = {m: [] for m in FSETS}
for idx in bb.samples:
    if yy[idx].sum() == 0: continue
    for m in FSETS: SB[m].append(pauc_80(yy[idx], P[m][idx]))
SB = {m: np.array(v) for m, v in SB.items()}
t = []
for a, b in [("M3_global_prior", "M1_global"), ("M2_prior", "M0_lesion"), ("M1_global", "M0_lesion"), ("M3_global_prior", "M0_lesion")]:
    d = SB[a] - SB[b]
    t.append(dict(comparison=f"{a} vs {b}", pooled_a=round(pauc_80(yy, P[a]), 4), pooled_b=round(pauc_80(yy, P[b]), 4),
                  delta=round(d.mean(), 4), ci95=f"[{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]", p=round((d <= 0).mean(), 4)))
loho_tests = pd.DataFrame(t); loho_tests["p_holm"] = holm(loho_tests.p.values).round(4)
print(loho_tests.to_string(index=False))

In [ ]:
lc_tab.to_csv(f"{WORK}/results_step5_learning_curve.csv"); lc_tests.to_csv(f"{WORK}/tests_step5_learning_curve.csv", index=False)
loho.to_csv(f"{WORK}/results_step5_loho.csv", index=False); loho_tests.to_csv(f"{WORK}/tests_step5_loho.csv", index=False)
print("saved. Download: results_step5_*.csv, tests_step5_*.csv, fig_learning_curve.png")